### Pre-Processing

In [ ]:
import pandas as pd
df = pd.read_csv("all_essays.csv")

df['Essay'] = df['Essay Question'] + ":" + df['Essay Answer']

df.head()

df.to_csv("essays.csv")

### Vanilla RNN

In [ ]:
 import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
import torch.nn as nn
import math
import torch.optim as optim

# Define the Vanilla RNN model
class VanillaRNNRegressor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(VanillaRNNRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, _ = self.rnn(embedded)
        # Take the last output of the RNN
        last_output = rnn_out[:, -1, :]
        # Pass through the fully connected layer
        out = self.fc(last_output)
        return out

# Load and preprocess data
df = pd.read_csv('essays.csv') 
texts = df['Essay'].values
scores = df['Score'].values

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
max_length = 500  # Adjust based on your needs
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Convert to PyTorch tensors
X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(scores, dtype=torch.float32).reshape(-1, 1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model parameters
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
hidden_dim = 64
learning_rate = 0.001
num_epochs = 100
batch_size = 32

# Initialize model, loss function, and optimizer
model = VanillaRNNRegressor(vocab_size, embedding_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # Process in batches
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i + batch_size]
        batch_y = y_train[i:i + batch_size]
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = total_loss / (len(X_train) / batch_size)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# Evaluate model
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    mae = mean_absolute_error(y_test.numpy(), test_predictions.numpy())
    mse = mean_squared_error(y_test.numpy(), test_predictions.numpy())
    rmse = math.sqrt(mse)
    
    print(f"\nTest Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")

# Save model and tokenizer
torch.save(model.state_dict(), 'vanilla_rnn_regressor.pth')
torch.save(tokenizer.to_json(), 'tokenizer_vanilla_rnn.json')
print("Model and tokenizer saved successfully!")

2025-02-24 10:10:13.888627: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-24 10:10:14.153864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740372014.252979 2519986 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740372014.281193 2519986 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-24 10:10:14.518880: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Epoch [5/100], Loss: 3.1726
Epoch [10/100], Loss: 3.1598
Epoch [15/100], Loss: 3.1566
Epoch [20/100], Loss: 3.1552
Epoch [25/100], Loss: 3.1545
Epoch [30/100], Loss: 3.1541
Epoch [35/100], Loss: 3.1539
Epoch [40/100], Loss: 3.1537
Epoch [45/100], Loss: 3.1536
Epoch [50/100], Loss: 3.1535
Epoch [55/100], Loss: 3.1534
Epoch [60/100], Loss: 3.1534
Epoch [65/100], Loss: 3.1534
Epoch [70/100], Loss: 3.1534
Epoch [75/100], Loss: 3.1533
Epoch [80/100], Loss: 3.1533
Epoch [85/100], Loss: 3.1533
Epoch [90/100], Loss: 3.1534
Epoch [95/100], Loss: 3.1534
Epoch [100/100], Loss: 3.1533

Test Metrics:
MSE: 3.4747
RMSE: 1.8641
MAE: 1.5564
Model and tokenizer saved successfully!


### BiRNN model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim

# Define the Bidirectional RNN model
class BiRNNRegressor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(BiRNNRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        # Note: We multiply hidden_dim by 2 because we're using bidirectional RNN
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        # Output shape: (batch_size, seq_length, hidden_dim * 2)
        rnn_out, _ = self.rnn(embedded)
        # Take the last output of the RNN from both directions
        last_output = rnn_out[:, -1, :]
        # Pass through the fully connected layer
        out = self.fc(last_output)
        return out

# Load and preprocess data
df = pd.read_csv('essays.csv') 
texts = df['Essay'].values
scores = df['Score'].values

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
max_length = 500  # Adjust based on your needs
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Convert to PyTorch tensors
X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(scores, dtype=torch.float32).reshape(-1, 1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model parameters
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
hidden_dim = 64
learning_rate = 0.001
num_epochs = 50
batch_size = 32

# Initialize model, loss function, and optimizer
model = BiRNNRegressor(vocab_size, embedding_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # Process in batches
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i + batch_size]
        batch_y = y_train[i:i + batch_size]
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = total_loss / (len(X_train) / batch_size)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# Evaluate model
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)

    mae = mean_absolute_error(y_test.numpy(), test_predictions.numpy())
    mse = mean_squared_error(y_test.numpy(), test_predictions.numpy())
    rmse = math.sqrt(mse)
    
    print(f"\nTest Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")

# Save model and tokenizer
torch.save(model.state_dict(), 'birnn_regressor.pth')
torch.save(tokenizer.to_json(), 'tokenizer_birnn.json')
print("Model and tokenizer saved successfully!")

Epoch [5/50], Loss: 3.2113
Epoch [10/50], Loss: 3.1750
Epoch [15/50], Loss: 3.1625
Epoch [20/50], Loss: 3.1567
Epoch [25/50], Loss: 3.1531
Epoch [30/50], Loss: 3.1506
Epoch [35/50], Loss: 3.1489
Epoch [40/50], Loss: 3.1477
Epoch [45/50], Loss: 3.1469
Epoch [50/50], Loss: 3.1463

Test Metrics:
MSE: 3.5133
RMSE: 1.8744
MAE: 1.5814
Model and tokenizer saved successfully!


### BiLSTM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim

# Define the LSTM model with more detailed architecture
class LSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(LSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers,
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        # LSTM returns output and (hidden_state, cell_state)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Concatenate the final hidden states from both directions
        final_hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        final_hidden = self.dropout(final_hidden)
        out = self.fc(final_hidden)
        return out

# Load and preprocess data
df = pd.read_csv('essays.csv')  
texts = df['Essay'].values
scores = df['Score'].values

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
max_length = 500  # Adjust based on your needs
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Convert to PyTorch tensors
X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(scores, dtype=torch.float32).reshape(-1, 1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model parameters
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
hidden_dim = 128
num_layers = 2  # Number of LSTM layers
dropout = 0.3   # Dropout rate
learning_rate = 0.001
num_epochs = 50
batch_size = 32

# Initialize model, loss function, and optimizer
model = LSTMRegressor(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout
)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # Process in batches
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i + batch_size]
        batch_y = y_train[i:i + batch_size]
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        loss.backward()
        # Add gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = total_loss / (len(X_train) / batch_size)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# Evaluate model
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    
    mae = mean_absolute_error(y_test.numpy(), test_predictions.numpy())
    mse = mean_squared_error(y_test.numpy(), test_predictions.numpy())
    rmse = math.sqrt(mse)
    
    print(f"\nTest Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")

# Save model and tokenizer
torch.save(model.state_dict(), 'lstm_regressor.pth')
torch.save(tokenizer.to_json(), 'tokenizer_lstm.json')
print("Model and tokenizer saved successfully!")

Epoch [5/50], Loss: 3.1781
Epoch [10/50], Loss: 2.7018
Epoch [15/50], Loss: 1.9917
Epoch [20/50], Loss: 1.5342
Epoch [25/50], Loss: 1.1725
Epoch [30/50], Loss: 0.8389
Epoch [35/50], Loss: 0.8492
Epoch [40/50], Loss: 0.5566
Epoch [45/50], Loss: 0.6205
Epoch [50/50], Loss: 0.6172

Test Metrics:
MSE: 5.0225
RMSE: 2.2411
MAE: 1.7492
Model and tokenizer saved successfully!


### LSTM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
import torch.nn as nn
import torch.optim as optim

# Define the LSTM model with more detailed architecture
class LSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(LSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers,
            batch_first=True, 
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        # LSTM returns output and (hidden_state, cell_state)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Concatenate the final hidden states from both directions
        final_hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        final_hidden = self.dropout(final_hidden)
        out = self.fc(final_hidden)
        return out

# Load and preprocess data
df = pd.read_csv('essays.csv')  
texts = df['Essay'].values
scores = df['Score'].values

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Pad sequences
max_length = 500  # Adjust based on your needs
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Convert to PyTorch tensors
X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(scores, dtype=torch.float32).reshape(-1, 1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model parameters
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
hidden_dim = 128
num_layers = 2  # Number of LSTM layers
dropout = 0.3   # Dropout rate
learning_rate = 0.001
num_epochs = 50
batch_size = 32

# Initialize model, loss function, and optimizer
model = LSTMRegressor(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout
)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # Process in batches
    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i + batch_size]
        batch_y = y_train[i:i + batch_size]
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        loss.backward()
        # Add gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = total_loss / (len(X_train) / batch_size)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

# Evaluate model
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    
    mae = mean_absolute_error(y_test.numpy(), test_predictions.numpy())
    mse = mean_squared_error(y_test.numpy(), test_predictions.numpy())
    rmse = math.sqrt(mse)
    
    print(f"\nTest Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")

# Save model and tokenizer
torch.save(model.state_dict(), 'lstm_regressor_false.pth')
torch.save(tokenizer.to_json(), 'tokenizer_lstm_false.json')
print("Model and tokenizer saved successfully!")

Epoch [5/50], Loss: 3.2547
Epoch [10/50], Loss: 3.2664
Epoch [15/50], Loss: 3.2055
Epoch [20/50], Loss: 3.2466
Epoch [25/50], Loss: 3.2333
Epoch [30/50], Loss: 3.2500
Epoch [35/50], Loss: 3.2146
Epoch [40/50], Loss: 3.2801
Epoch [45/50], Loss: 3.2440
Epoch [50/50], Loss: 3.2741

Test Metrics:
MSE: 3.4460
RMSE: 1.8563
MAE: 1.5738
Model and tokenizer saved successfully!


### ANN with engineered features

In [ ]:
import spacy
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Load models
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Reuse the existing feature extraction functions
def calculate_topic_relevance(essay, topic):
    essay_embedding = sbert_model.encode(essay)
    topic_embedding = sbert_model.encode(topic)
    similarity = np.dot(essay_embedding, topic_embedding) / (
        np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
    )
    return similarity

def detect_poetic_justice(essay):
    doc = nlp(essay)
    justice_keywords = ["deserve", "deserved", "justice", "punished", "rewarded", "consequences", "virtue", "vice"]
    justice_count = sum(1 for token in doc if token.text.lower() in justice_keywords)
    
    sentences = list(doc.sents)
    if len(sentences) > 1:
        sentiment_scores = [sentiment_analyzer.polarity_scores(sent.text)['compound'] for sent in sentences]
        sentiment_shift = max(sentiment_scores) - min(sentiment_scores)
    else:
        sentiment_shift = 0
    
    causal_phrases = ["because", "due to", "as a result"]
    causal_count = sum(1 for token in doc if token.text.lower() in causal_phrases)
    
    poetic_justice_score = justice_count + sentiment_shift + causal_count
    return poetic_justice_score

def extract_features_with_topic(essay, topic):
    doc = nlp(essay)

    word_count = len([token.text for token in doc if token.is_alpha])
    sentence_count = len(list(doc.sents))
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
    unique_words = len(set(token.text.lower() for token in doc if token.is_alpha))
    lexical_density = unique_words / word_count if word_count > 0 else 0
    ttr = unique_words / word_count if word_count > 0 else 0
    
    pos_count = {pos: len([token for token in doc if token.pos_ == pos]) for pos in ['NOUN', 'VERB']}
    
    flesch_kincaid = textstat.flesch_kincaid_grade(essay)
    dale_chall = textstat.dale_chall_readability_score(essay)
    
    sentiment_scores = sentiment_analyzer.polarity_scores(essay)
    sentiment = sentiment_scores['compound']
    
    sentences = [sent.text for sent in doc.sents]
    if len(sentences) > 1:
        embeddings = sbert_model.encode(sentences)
        coherence = np.mean([
            np.dot(embeddings[i], embeddings[i + 1]) /
            (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]))
            for i in range(len(embeddings) - 1)
        ])
    else:
        coherence = 0

    poetic_justice = detect_poetic_justice(essay)
    topic_relevance = calculate_topic_relevance(essay, topic)

    features = {
        'word_count': word_count,
        'avg_sentence_length': avg_sentence_length,
        'unique_words': unique_words,
        'lexical_density': lexical_density,
        'ttr': ttr,
        'nouns': pos_count['NOUN'],
        'verbs': pos_count['VERB'],
        'flesch_kincaid': flesch_kincaid,
        'dale_chall': dale_chall,
        'sentiment': sentiment,
        'coherence': coherence,
        'poetic_justice': poetic_justice,
        'topic_relevance': topic_relevance
    }

    return features

# Define the ANN model
class EssayScorer(nn.Module):
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.relu(self.layer3(x))
        x = self.output(x)
        return x

# Load and prepare data
df = pd.read_csv('essays.csv')

# Extract features
engineered_features = [
    extract_features_with_topic(essay, topic) 
    for essay, topic in zip(df['Essay Answer'], df['Essay Question'])
]
X = pd.DataFrame(engineered_features)
y = df['Score']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train.values)
X_test_tensor = torch.FloatTensor(X_test.values)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

# Initialize model, loss function, and optimizer
model = EssayScorer(input_size=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
batch_size = 32

for epoch in range(num_epochs):
    model.train()
    for i in range(0, len(X_train_tensor), batch_size):
        batch_X = X_train_tensor[i:i + batch_size]
        batch_y = y_train_tensor[i:i + batch_size]
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

# Evaluate model
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor)
    
    # Calculate metrics
    mse = mean_squared_error(y_test_tensor.numpy(), y_pred.numpy())
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test_tensor.numpy(), y_pred.numpy())
    
    print(f"\nTest Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")

# Save model
torch.save(model.state_dict(), 'ann_essay_scorer_without_zeros.pth')
print("\nANN Model saved successfully!")

/home/shivam/.conda/envs/new_env_test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [5/50], Loss: 1.9021
Epoch [10/50], Loss: 2.0441
Epoch [15/50], Loss: 2.6617
Epoch [20/50], Loss: 2.6540
Epoch [25/50], Loss: 2.0798
Epoch [30/50], Loss: 1.4476
Epoch [35/50], Loss: 1.7374
Epoch [40/50], Loss: 1.8031
Epoch [45/50], Loss: 1.9720
Epoch [50/50], Loss: 2.1471

Test Metrics:
MSE: 3.8228
RMSE: 1.9552
MAE: 1.6938

ANN Model saved successfully!


### ANN with GLoVe

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
import numpy as np
from typing import Dict, List
import requests
import os
import zipfile
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class GloVeEmbeddings:
    def __init__(self, embedding_dim=300):
        self.embedding_dim = embedding_dim
        self.word_to_vec = {}
        
    def download_glove(self):
        """
        Download GloVe embeddings if not already present
        """
        glove_path = f'glove.6B.{self.embedding_dim}d.txt'
        
        if not os.path.exists(glove_path):
            print("Downloading GloVe embeddings...")
            url = "http://nlp.stanford.edu/data/glove.6B.zip"
            response = requests.get(url, stream=True)
            total_size = int(response.headers.get('content-length', 0))
            
            with open("glove.6B.zip", 'wb') as file, tqdm(
                desc="Downloading",
                total=total_size,
                unit='iB',
                unit_scale=True
            ) as pbar:
                for data in response.iter_content(chunk_size=1024):
                    size = file.write(data)
                    pbar.update(size)
            
            print("Extracting GloVe embeddings...")
            with zipfile.ZipFile("glove.6B.zip", 'r') as zip_ref:
                zip_ref.extract(f'glove.6B.{self.embedding_dim}d.txt')
            
            os.remove("glove.6B.zip")
    
    def load_embeddings(self):
        """
        Load GloVe embeddings into memory
        """
        print(f"Loading {self.embedding_dim}-dimensional GloVe embeddings...")
        glove_path = f'glove.6B.{self.embedding_dim}d.txt'
        
        with open(glove_path, 'r', encoding='utf-8') as f:
            for line in tqdm(f, desc="Loading embeddings"):
                values = line.split()
                word = values[0]
                vector = np.asarray(values[1:], dtype='float32')
                self.word_to_vec[word] = vector
        
        print(f"Loaded {len(self.word_to_vec)} word vectors")
    
    def get_word_embedding(self, word: str) -> np.ndarray:
        """
        Get embedding for a single word
        """
        word = word.lower()
        return self.word_to_vec.get(word, np.zeros(self.embedding_dim))
    
    def get_text_embedding(self, text: str) -> np.ndarray:
        """
        Get embedding for a piece of text by averaging word embeddings
        """
        words = text.lower().split()
        word_embeddings = [self.get_word_embedding(word) for word in words]
        
        if not word_embeddings:
            return np.zeros(self.embedding_dim)
        
        return np.mean(word_embeddings, axis=0)

class EssayRegressor(nn.Module):
    def __init__(self, embedding_dim: int = 300):
        super(EssayRegressor, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

def prepare_data(csv_path: str, glove: GloVeEmbeddings):
    """
    Prepare data using GloVe embeddings
    """
    print("Loading and preparing data...")
    df = pd.read_csv(csv_path)
    
    # Get embeddings for essays
    print("Converting essays to GloVe embeddings...")
    essay_embeddings = []
    
    for essay in tqdm(df['Essay'], desc="Processing essays"):
        embedding = glove.get_text_embedding(essay)
        essay_embeddings.append(embedding)
    
    essay_embeddings = np.array(essay_embeddings)
    scores = df['Score'].values
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        essay_embeddings, scores, test_size=0.2, random_state=42
    )
    
    # Convert to tensors
    X_train = torch.FloatTensor(X_train)
    X_test = torch.FloatTensor(X_test)
    y_train = torch.FloatTensor(y_train).reshape(-1, 1)
    y_test = torch.FloatTensor(y_test).reshape(-1, 1)
    
    print(f"\nData shapes:")
    print(f"Training data: {X_train.shape}")
    print(f"Test data: {X_test.shape}")
    
    # Show sample of embeddings
    print("\nSample embedding for first essay:")
    print(f"First 10 dimensions: {X_train[0][:10]}")
    print(f"Mean value: {X_train[0].mean():.4f}")
    print(f"Std deviation: {X_train[0].std():.4f}")
    
    return X_train, X_test, y_train, y_test

def train_model(model, X_train, y_train, X_test, y_test, 
                num_epochs: int = 50, batch_size: int = 32, 
                learning_rate: float = 0.001):
    """
    Train the model and print metrics
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print("\nStarting training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        for i in range(0, len(X_train), batch_size):
            batch_X = X_train[i:i + batch_size]
            batch_y = y_train[i:i + batch_size]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / num_batches
            print(f"Epoch [{epoch + 1}/{num_epochs}], Average Loss: {avg_loss:.4f}")
    
    # Evaluate model
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test)
        
        mse = mean_squared_error(y_test.numpy(), y_pred.numpy())
        rmse = math.sqrt(mse)
        mae = mean_absolute_error(y_test.numpy(), y_pred.numpy())
        
        print("\nTest Metrics:")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
    
    return model

def main():
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Initialize and load GloVe embeddings
    glove = GloVeEmbeddings(embedding_dim=300)
    glove.download_glove()
    glove.load_embeddings()
    
    # Prepare data
    X_train, X_test, y_train, y_test = prepare_data(
        'essays.csv',
        glove
    )
    
    # Initialize and train model
    model = EssayRegressor()
    model = train_model(model, X_train, y_train, X_test, y_test)
    
    # Save model
    torch.save(model.state_dict(), 'glove_essay_regressor.pth')
    print("\nModel saved successfully!")

if __name__ == "__main__":
    main()

Downloading: 100%|██████████| 862M/862M [03:04<00:00, 4.66MiB/s]  


Extracting GloVe embeddings...
Loading 300-dimensional GloVe embeddings...


Loading embeddings: 400000it [00:13, 29335.51it/s]


Loaded 400000 word vectors
Loading and preparing data...
Converting essays to GloVe embeddings...


Processing essays: 100%|██████████| 2979/2979 [00:00<00:00, 5021.26it/s]



Data shapes:
Training data: torch.Size([2383, 300])
Test data: torch.Size([596, 300])

Sample embedding for first essay:
First 10 dimensions: tensor([-0.0652,  0.0775, -0.0238, -0.1464, -0.0321,  0.0097, -0.0252,  0.1594,
         0.0403, -1.4992])
Mean value: -0.0044
Std deviation: 0.1747

Starting training...
Epoch [5/50], Average Loss: 3.5844
Epoch [10/50], Average Loss: 3.3967
Epoch [15/50], Average Loss: 3.2953
Epoch [20/50], Average Loss: 3.1753
Epoch [25/50], Average Loss: 3.1715
Epoch [30/50], Average Loss: 3.2551
Epoch [35/50], Average Loss: 3.1197
Epoch [40/50], Average Loss: 3.0078
Epoch [45/50], Average Loss: 3.0270
Epoch [50/50], Average Loss: 2.9600

Test Metrics:
MSE: 3.3722
RMSE: 1.8363
MAE: 1.5379

Model saved successfully!


### ANN with Engineered Features (Normalized)

In [ ]:
import spacy
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math

# Load models
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

class FeatureNormalizer:
    def __init__(self):
        self.scalers = {}
        
    def fit(self, feature_df):
        """Fit scalers for each feature"""
        for column in feature_df.columns:
            self.scalers[column] = MinMaxScaler(feature_range=(1, 10))
            self.scalers[column].fit(feature_df[column].values.reshape(-1, 1))
    
    def transform(self, feature_df):
        """Transform features to range 1-10"""
        normalized_df = pd.DataFrame()
        for column in feature_df.columns:
            normalized_values = self.scalers[column].transform(
                feature_df[column].values.reshape(-1, 1)
            )
            normalized_df[column] = normalized_values.flatten()
        return normalized_df
    
    def fit_transform(self, feature_df):
        """Fit and transform features"""
        self.fit(feature_df)
        return self.transform(feature_df)

# Reuse existing feature extraction functions
def calculate_topic_relevance(essay, topic):
    essay_embedding = sbert_model.encode(essay)
    topic_embedding = sbert_model.encode(topic)
    similarity = np.dot(essay_embedding, topic_embedding) / (
        np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
    )
    return similarity

def detect_poetic_justice(essay):
    doc = nlp(essay)
    justice_keywords = ["deserve", "deserved", "justice", "punished", "rewarded", "consequences", "virtue", "vice"]
    justice_count = sum(1 for token in doc if token.text.lower() in justice_keywords)
    
    sentences = list(doc.sents)
    if len(sentences) > 1:
        sentiment_scores = [sentiment_analyzer.polarity_scores(sent.text)['compound'] for sent in sentences]
        sentiment_shift = max(sentiment_scores) - min(sentiment_scores)
    else:
        sentiment_shift = 0
    
    causal_phrases = ["because", "due to", "as a result"]
    causal_count = sum(1 for token in doc if token.text.lower() in causal_phrases)
    
    poetic_justice_score = justice_count + sentiment_shift + causal_count
    return poetic_justice_score

def extract_features_with_topic(essay, topic):
    doc = nlp(essay)

    word_count = len([token.text for token in doc if token.is_alpha])
    sentence_count = len(list(doc.sents))
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
    unique_words = len(set(token.text.lower() for token in doc if token.is_alpha))
    lexical_density = unique_words / word_count if word_count > 0 else 0
    ttr = unique_words / word_count if word_count > 0 else 0
    
    pos_count = {pos: len([token for token in doc if token.pos_ == pos]) for pos in ['NOUN', 'VERB']}
    
    flesch_kincaid = textstat.flesch_kincaid_grade(essay)
    dale_chall = textstat.dale_chall_readability_score(essay)
    
    sentiment_scores = sentiment_analyzer.polarity_scores(essay)
    sentiment = sentiment_scores['compound']
    
    sentences = [sent.text for sent in doc.sents]
    if len(sentences) > 1:
        embeddings = sbert_model.encode(sentences)
        coherence = np.mean([
            np.dot(embeddings[i], embeddings[i + 1]) /
            (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]))
            for i in range(len(embeddings) - 1)
        ])
    else:
        coherence = 0

    poetic_justice = detect_poetic_justice(essay)
    topic_relevance = calculate_topic_relevance(essay, topic)

    features = {
        'word_count': word_count,
        'avg_sentence_length': avg_sentence_length,
        'unique_words': unique_words,
        'lexical_density': lexical_density,
        'ttr': ttr,
        'nouns': pos_count['NOUN'],
        'verbs': pos_count['VERB'],
        'flesch_kincaid': flesch_kincaid,
        'dale_chall': dale_chall,
        'sentiment': sentiment,
        'coherence': coherence,
        'poetic_justice': poetic_justice,
        'topic_relevance': topic_relevance
    }

    return features

class EssayScorer(nn.Module):
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.relu(self.layer3(x))
        x = self.output(x)
        return x

def main():
    # Load data
    print("Loading data...")
    df = pd.read_csv('essays.csv')

    # Extract features
    print("Extracting features...")
    engineered_features = [
        extract_features_with_topic(essay, topic) 
        for essay, topic in zip(df['Essay Answer'], df['Essay Question'])
    ]
    X = pd.DataFrame(engineered_features)
    y = df['Score']

    # Initialize feature normalizer
    print("Normalizing features to range 1-10...")
    normalizer = FeatureNormalizer()

    # Train-test split before normalization
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Fit normalizer on training data and transform both sets
    X_train_normalized = normalizer.fit_transform(X_train)
    X_test_normalized = normalizer.transform(X_test)

    # Print feature ranges for verification
    print("\nFeature ranges after normalization (training set):")
    for column in X_train_normalized.columns:
        min_val = X_train_normalized[column].min()
        max_val = X_train_normalized[column].max()
        mean_val = X_train_normalized[column].mean()
        print(f"{column:20} - Min: {min_val:.2f}, Max: {max_val:.2f}, Mean: {mean_val:.2f}")

    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train_normalized.values)
    X_test_tensor = torch.FloatTensor(X_test_normalized.values)
    y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
    y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

    # Initialize model, loss function, and optimizer
    model = EssayScorer(input_size=X_train.shape[1])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training loop
    print("\nStarting training...")
    num_epochs = 500
    batch_size = 32

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        for i in range(0, len(X_train_tensor), batch_size):
            batch_X = X_train_tensor[i:i + batch_size]
            batch_y = y_train_tensor[i:i + batch_size]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / num_batches
            print(f"Epoch [{epoch + 1}/{num_epochs}], Average Loss: {avg_loss:.4f}")

    # Evaluate model
    model.eval()
    with torch.no_grad():
        y_pred = model(X_test_tensor)
        
        mse = mean_squared_error(y_test_tensor.numpy(), y_pred.numpy())
        rmse = math.sqrt(mse)
        mae = mean_absolute_error(y_test_tensor.numpy(), y_pred.numpy())
        
        print("\nTest Metrics:")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")

    # Save model and normalizer
    torch.save({
        'model_state_dict': model.state_dict(),
        'normalizer': normalizer,
        'feature_columns': list(X_train.columns)
    }, 'normalized_essay_scorer.pth')
    print("\nModel and normalizer saved successfully!")

if __name__ == "__main__":
    main()

Loading data...
Extracting features...
Normalizing features to range 1-10...

Feature ranges after normalization (training set):
word_count           - Min: 1.00, Max: 10.00, Mean: 4.23
avg_sentence_length  - Min: 1.00, Max: 10.00, Mean: 1.85
unique_words         - Min: 1.00, Max: 10.00, Mean: 4.89
lexical_density      - Min: 1.00, Max: 10.00, Mean: 4.22
ttr                  - Min: 1.00, Max: 10.00, Mean: 4.22
nouns                - Min: 1.00, Max: 10.00, Mean: 4.16
verbs                - Min: 1.00, Max: 10.00, Mean: 3.51
flesch_kincaid       - Min: 1.00, Max: 10.00, Mean: 1.84
dale_chall           - Min: 1.00, Max: 10.00, Mean: 3.14
sentiment            - Min: 1.00, Max: 10.00, Mean: 8.66
coherence            - Min: 1.00, Max: 10.00, Mean: 5.83
poetic_justice       - Min: 1.00, Max: 10.00, Mean: 1.82
topic_relevance      - Min: 1.00, Max: 10.00, Mean: 7.28

Starting training...
Epoch [5/500], Average Loss: 3.4735
Epoch [10/500], Average Loss: 3.1619
Epoch [15/500], Average Loss: 3.124

### ANN with GloVe embeddings with Topic and Essay as different Embedding

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
from typing import Dict, List, Tuple

class EnhancedEssayRegressor(nn.Module):
    def __init__(self, embedding_dim: int = 300):
        super(EnhancedEssayRegressor, self).__init__()
        
        # Process question embedding
        self.question_network = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Process answer embedding
        self.answer_network = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Combined network including relevance score
        self.combined_network = nn.Sequential(
            nn.Linear(128 * 2 + 1, 128),  # +1 for relevance score
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(64, 1)
        )

    def forward(self, question_embedding, answer_embedding, relevance_score):
        # Process embeddings separately
        question_features = self.question_network(question_embedding)
        answer_features = self.answer_network(answer_embedding)
        
        # Combine features with relevance score
        combined = torch.cat([
            question_features, 
            answer_features, 
            relevance_score.unsqueeze(1)
        ], dim=1)
        
        return self.combined_network(combined)

def calculate_topic_relevance(essay_embedding: np.ndarray, topic_embedding: np.ndarray) -> float:
    """Calculate cosine similarity between essay and topic embeddings"""
    similarity = np.dot(essay_embedding, topic_embedding) / (
        np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
    )
    return similarity

def prepare_enhanced_data(csv_path: str, glove: GloVeEmbeddings) -> Tuple:
    """Prepare data with separate question and answer embeddings"""
    print("Loading and preparing data...")
    df = pd.read_csv(csv_path)
    
    question_embeddings = []
    answer_embeddings = []
    relevance_scores = []
    
    print("Processing questions and answers...")
    for idx in tqdm(range(len(df))):
        # Get embeddings
        question_emb = glove.get_text_embedding(df['Essay Question'].iloc[idx])
        answer_emb = glove.get_text_embedding(df['Essay Answer'].iloc[idx])
        
        # Calculate relevance score
        relevance = calculate_topic_relevance(answer_emb, question_emb)
        
        question_embeddings.append(question_emb)
        answer_embeddings.append(answer_emb)
        relevance_scores.append(relevance)
    
    # Convert to numpy arrays
    question_embeddings = np.array(question_embeddings)
    answer_embeddings = np.array(answer_embeddings)
    relevance_scores = np.array(relevance_scores)
    scores = df['Score'].values
    
    # Split data
    indices = range(len(scores))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
    
    # Create train and test data dictionaries
    train_data = {
        'question': torch.FloatTensor(question_embeddings[train_idx]),
        'answer': torch.FloatTensor(answer_embeddings[train_idx]),
        'relevance': torch.FloatTensor(relevance_scores[train_idx]),
        'labels': torch.FloatTensor(scores[train_idx]).reshape(-1, 1)
    }
    
    test_data = {
        'question': torch.FloatTensor(question_embeddings[test_idx]),
        'answer': torch.FloatTensor(answer_embeddings[test_idx]),
        'relevance': torch.FloatTensor(relevance_scores[test_idx]),
        'labels': torch.FloatTensor(scores[test_idx]).reshape(-1, 1)
    }
    
    return train_data, test_data

def train_enhanced_model(model, train_data: Dict, test_data: Dict, 
                        num_epochs: int = 500, batch_size: int = 32, 
                        learning_rate: float = 0.001):
    """Train the enhanced model with separate embeddings and relevance scores"""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    print("\nStarting training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        
        for i in range(0, len(train_data['question']), batch_size):
            batch_q = train_data['question'][i:i + batch_size]
            batch_a = train_data['answer'][i:i + batch_size]
            batch_r = train_data['relevance'][i:i + batch_size]
            batch_y = train_data['labels'][i:i + batch_size]
            
            optimizer.zero_grad()
            outputs = model(batch_q, batch_a, batch_r)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / num_batches
            print(f"Epoch [{epoch + 1}/{num_epochs}], Average Loss: {avg_loss:.4f}")
    
    # Evaluate model
    model.eval()
    with torch.no_grad():
        y_pred = model(
            test_data['question'],
            test_data['answer'],
            test_data['relevance']
        )
        
        mse = mean_squared_error(test_data['labels'].numpy(), y_pred.numpy())
        rmse = math.sqrt(mse)
        mae = mean_absolute_error(test_data['labels'].numpy(), y_pred.numpy())
        
        print("\nTest Metrics:")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
    
    return model

def main():
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Initialize and load GloVe embeddings
    glove = GloVeEmbeddings(embedding_dim=300)
    glove.download_glove()
    glove.load_embeddings()
    
    # Prepare enhanced data
    train_data, test_data = prepare_enhanced_data(
        'essays.csv',
        glove
    )
    
    # Initialize and train enhanced model
    model = EnhancedEssayRegressor()
    model = train_enhanced_model(model, train_data, test_data)
    
    # Save model
    torch.save(model.state_dict(), 'enhanced_essay_regressor.pth')
    print("\nModel saved successfully!")

if __name__ == "__main__":
    main()

Loading 300-dimensional GloVe embeddings...


Loading embeddings: 400000it [00:13, 29290.56it/s]


Loaded 400000 word vectors
Loading and preparing data...
Processing questions and answers...


100%|██████████| 2979/2979 [00:00<00:00, 4108.42it/s]



Starting training...
Epoch [5/500], Average Loss: 3.5195
Epoch [10/500], Average Loss: 3.3003
Epoch [15/500], Average Loss: 3.1647
Epoch [20/500], Average Loss: 3.2454
Epoch [25/500], Average Loss: 3.1515
Epoch [30/500], Average Loss: 3.1037
Epoch [35/500], Average Loss: 3.1029
Epoch [40/500], Average Loss: 2.9768
Epoch [45/500], Average Loss: 2.8594
Epoch [50/500], Average Loss: 2.9151
Epoch [55/500], Average Loss: 2.8455
Epoch [60/500], Average Loss: 2.8347
Epoch [65/500], Average Loss: 2.7915
Epoch [70/500], Average Loss: 2.6498
Epoch [75/500], Average Loss: 2.7105
Epoch [80/500], Average Loss: 2.6531
Epoch [85/500], Average Loss: 2.5833
Epoch [90/500], Average Loss: 2.6137
Epoch [95/500], Average Loss: 2.4838
Epoch [100/500], Average Loss: 2.4048
Epoch [105/500], Average Loss: 2.4364
Epoch [110/500], Average Loss: 2.3786
Epoch [115/500], Average Loss: 2.3520
Epoch [120/500], Average Loss: 2.2954
Epoch [125/500], Average Loss: 2.2964
Epoch [130/500], Average Loss: 2.2189
Epoch [135

### ANN using Topic and Essay Embeddings along with SpellCheck and Grammer included

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"  # Changed to true for batch processing

import spacy
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
from spellchecker import SpellChecker
import re
from tqdm import tqdm
import warnings
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import language_tool_python

# Configure warnings and logging
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
logging.basicConfig(level=logging.ERROR)

# Initialize shared resources
print("Loading NLP models...")
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
spell_checker = SpellChecker()
language_tool = language_tool_python.LanguageTool('en-US')
print("Models loaded successfully!")

def batch_process_essays(essays, func, batch_size=10, desc="Processing"):
    """Process essays in batches to improve performance"""
    results = []
    for i in tqdm(range(0, len(essays), batch_size), desc=desc):
        batch = essays[i:i + batch_size]
        batch_results = [func(essay) for essay in batch]
        results.extend(batch_results)
    return results

def calculate_error_penalties(essay):
    """Optimized error penalty calculation"""
    try:
        cleaned_text = re.sub(r'\s+', ' ', essay).strip()
        words = cleaned_text.split()
        
        if not words:
            return {
                'spelling_penalty': 1.0,
                'grammar_penalty': 1.0,
                'total_penalty': 1.0
            }
        
        # Batch process spelling checks
        misspelled = spell_checker.unknown(words)
        spelling_errors = len(misspelled)
        
        # Limit grammar check to first 1000 characters to improve speed
        truncated_text = cleaned_text[:1000]
        grammar_matches = language_tool.check(truncated_text)
        grammar_errors = len(grammar_matches)
        
        word_count = len(words)
        spelling_penalty = min(1.0, spelling_errors / (word_count * 0.1))
        grammar_penalty = min(1.0, grammar_errors / (word_count * 0.05))
        total_penalty = (spelling_penalty * 0.6) + (grammar_penalty * 0.4)
        
        return {
            'spelling_penalty': spelling_penalty,
            'grammar_penalty': grammar_penalty,
            'total_penalty': total_penalty
        }
    except Exception as e:
        logging.error(f"Error in calculate_error_penalties: {str(e)}")
        return {'spelling_penalty': 0.5, 'grammar_penalty': 0.5, 'total_penalty': 0.5}

def batch_encode_texts(texts, batch_size=32):
    """Batch encode texts using sentence transformer"""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embeddings = sbert_model.encode(batch)
        all_embeddings.extend(embeddings)
    return all_embeddings

def extract_features_with_topic_and_errors(essay, topic):
    """Optimized feature extraction"""
    try:
        # Process with spaCy
        doc = nlp(essay)
        
        # Basic text statistics
        word_count = len([token.text for token in doc if token.is_alpha])
        sentence_count = len(list(doc.sents))
        avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
        unique_words = len(set(token.text.lower() for token in doc if token.is_alpha))
        lexical_density = unique_words / word_count if word_count > 0 else 0
        ttr = unique_words / word_count if word_count > 0 else 0
        
        # POS counting
        pos_count = {pos: len([token for token in doc if token.pos_ == pos]) for pos in ['NOUN', 'VERB']}
        
        # Readability scores (cached internally by textstat)
        flesch_kincaid = textstat.flesch_kincaid_grade(essay)
        dale_chall = textstat.dale_chall_readability_score(essay)
        
        # Sentiment analysis
        sentiment = sentiment_analyzer.polarity_scores(essay)['compound']
        
        # Simplified coherence calculation
        sentences = [sent.text for sent in doc.sents]
        if len(sentences) > 1:
            # Only process first 5 sentences for speed
            sentences = sentences[:5]
            embeddings = sbert_model.encode(sentences)
            coherence_scores = [
                np.dot(embeddings[i], embeddings[i + 1]) /
                (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]))
                for i in range(len(embeddings) - 1)
            ]
            coherence = np.mean(coherence_scores) if coherence_scores else 0
        else:
            coherence = 0
        
        # Calculate topic relevance
        essay_embedding = sbert_model.encode([essay])[0]
        topic_embedding = sbert_model.encode([topic])[0]
        topic_relevance = np.dot(essay_embedding, topic_embedding) / (
            np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
        )
        
        # Get error metrics
        error_metrics = calculate_error_penalties(essay)
        
        return {
            'word_count': word_count,
            'avg_sentence_length': avg_sentence_length,
            'unique_words': unique_words,
            'lexical_density': lexical_density,
            'ttr': ttr,
            'nouns': pos_count['NOUN'],
            'verbs': pos_count['VERB'],
            'flesch_kincaid': flesch_kincaid,
            'dale_chall': dale_chall,
            'sentiment': sentiment,
            'coherence': coherence,
            'topic_relevance': topic_relevance,
            'spelling_penalty': error_metrics['spelling_penalty'],
            'grammar_penalty': error_metrics['grammar_penalty'],
            'total_error_penalty': error_metrics['total_penalty']
        }
    except Exception as e:
        logging.error(f"Error in feature extraction: {str(e)}")
        return {feature: 0 for feature in [
            'word_count', 'avg_sentence_length', 'unique_words', 'lexical_density',
            'ttr', 'nouns', 'verbs', 'flesch_kincaid', 'dale_chall', 'sentiment',
            'coherence', 'topic_relevance', 'spelling_penalty',
            'grammar_penalty', 'total_error_penalty'
        ]}

def process_essay_batch(args):
    """Process a batch of essays"""
    essays, topics = args
    return [extract_features_with_topic_and_errors(essay, topic) 
            for essay, topic in zip(essays, topics)]

def main():
    try:
        # Load data
        print("Loading data...")
        df = pd.read_csv('essays.csv')
        total_essays = len(df)
        print(f"Loaded {total_essays} essays successfully!")

        # Process essays in parallel batches
        print("\nExtracting features...")
        batch_size = 10
        num_workers = min(os.cpu_count(), 4)  # Limit to 4 workers max
        
        essay_batches = [
            (df['Essay Answer'][i:i + batch_size].tolist(),
             df['Essay Question'][i:i + batch_size].tolist())
            for i in range(0, len(df), batch_size)
        ]
        
        all_features = []
        processed_essays = 0
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(process_essay_batch, batch) 
                      for batch in essay_batches]
            
            progress_bar = tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Processing essay batches"
            )
            
            for future in progress_bar:
                batch_features = future.result()
                all_features.extend(batch_features)
                processed_essays += len(batch_features)
                progress_bar.set_postfix({
                    'essays_processed': f'{processed_essays}/{total_essays}',
                    'completion': f'{(processed_essays/total_essays)*100:.1f}%'
                })
        
        print(f"\nProcessed {processed_essays} essays in {len(essay_batches)} batches")
        
        X = pd.DataFrame(all_features)
        y = df['Score']

        # Initialize feature normalizer
        print("\nNormalizing features...")
        normalizer = FeatureNormalizer()

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Normalize features
        X_train_normalized = normalizer.fit_transform(X_train)
        X_test_normalized = normalizer.transform(X_test)

        # Print feature ranges
        print("\nFeature ranges after normalization (training set):")
        for column in X_train_normalized.columns:
            min_val = X_train_normalized[column].min()
            max_val = X_train_normalized[column].max()
            mean_val = X_train_normalized[column].mean()
            print(f"{column:20} - Min: {min_val:.2f}, Max: {max_val:.2f}, Mean: {mean_val:.2f}")

        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train_normalized.values)
        X_test_tensor = torch.FloatTensor(X_test_normalized.values)
        y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
        y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

        # Initialize model
        model = EssayScorer(input_size=X_train.shape[1])
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Training loop
        print("\nStarting training...")
        num_epochs = 500
        batch_size = 32
        
        progress_bar = tqdm(range(num_epochs), desc="Training")
        for epoch in progress_bar:
            model.train()
            total_loss = 0
            num_batches = 0
            
            for i in range(0, len(X_train_tensor), batch_size):
                batch_X = X_train_tensor[i:i + batch_size]
                batch_y = y_train_tensor[i:i + batch_size]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                num_batches += 1
            
            avg_loss = total_loss / num_batches
            progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})

        # Evaluate model
        print("\nEvaluating model...")
        model.eval()
        with torch.no_grad():
            y_pred = model(X_test_tensor)
            
            mse = mean_squared_error(y_test_tensor.numpy(), y_pred.numpy())
            rmse = math.sqrt(mse)
            mae = mean_absolute_error(y_test_tensor.numpy(), y_pred.numpy())
            
            print("\nTest Metrics:")
            print(f"MSE: {mse:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f"MAE: {mae:.4f}")

        # Save model
        print("\nSaving model...")
        torch.save({
            'model_state_dict': model.state_dict(),
            'normalizer': normalizer,
            'feature_columns': list(X_train.columns)
        }, 'normalized_essay_scorer.pth')
        print("Model and normalizer saved successfully!")

    except Exception as e:
        logging.error(f"Error in main function: {str(e)}")
        raise

if __name__ == "__main__":
    main()

Loading NLP models...
Models loaded successfully!
Loading data...
Loaded 2979 essays successfully!

Extracting features...


Processing essay batches: 100%|██████████| 298/298 [02:35<00:00,  1.92it/s, essays_processed=2979/2979, completion=100.0%]



Processed 2979 essays in 298 batches

Normalizing features...


Transforming features: 100%|██████████| 15/15 [00:00<00:00, 2906.79it/s]



Feature ranges after normalization (training set):
word_count           - Min: 1.00, Max: 10.00, Mean: 4.23
avg_sentence_length  - Min: 1.00, Max: 10.00, Mean: 1.85
unique_words         - Min: 1.00, Max: 10.00, Mean: 4.89
lexical_density      - Min: 1.00, Max: 10.00, Mean: 4.23
ttr                  - Min: 1.00, Max: 10.00, Mean: 4.23
nouns                - Min: 1.00, Max: 10.00, Mean: 4.15
verbs                - Min: 1.00, Max: 10.00, Mean: 3.51
flesch_kincaid       - Min: 1.00, Max: 10.00, Mean: 1.85
dale_chall           - Min: 1.00, Max: 10.00, Mean: 3.10
sentiment            - Min: 1.00, Max: 10.00, Mean: 8.63
coherence            - Min: 1.00, Max: 10.00, Mean: 5.97
topic_relevance      - Min: 1.00, Max: 10.00, Mean: 7.28
spelling_penalty     - Min: 1.00, Max: 10.00, Mean: 8.97
grammar_penalty      - Min: 1.00, Max: 10.00, Mean: 7.42
total_error_penalty  - Min: 1.00, Max: 10.00, Mean: 7.71

Starting training...


Training: 100%|██████████| 500/500 [01:59<00:00,  4.20it/s, loss=2.4493]


Evaluating model...

Test Metrics:
MSE: 3.4415
RMSE: 1.8551
MAE: 1.5400

Saving model...
Model and normalizer saved successfully!


### ANN with Topic and Essay Embeddings with SpellCheck, Grammer and newly Graded Scores

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"  # Changed to true for batch processing

import spacy
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
from spellchecker import SpellChecker
import re
from tqdm import tqdm
import warnings
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import language_tool_python

# Configure warnings and logging
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
logging.basicConfig(level=logging.ERROR)

# Initialize shared resources
print("Loading NLP models...")
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
spell_checker = SpellChecker()
language_tool = language_tool_python.LanguageTool('en-US')
print("Models loaded successfully!")

class FeatureNormalizer:
    def __init__(self):
        self.scalers = {}
        
    def fit(self, feature_df):
        """Fit scalers for each feature"""
        for column in feature_df.columns:
            self.scalers[column] = MinMaxScaler(feature_range=(1, 10))
            self.scalers[column].fit(feature_df[column].values.reshape(-1, 1))
    
    def transform(self, feature_df):
        """Transform features to range 1-10"""
        normalized_df = pd.DataFrame()
        for column in feature_df.columns:
            normalized_values = self.scalers[column].transform(
                feature_df[column].values.reshape(-1, 1)
            )
            normalized_df[column] = normalized_values.flatten()
        return normalized_df
    
    def fit_transform(self, feature_df):
        """Fit and transform features"""
        self.fit(feature_df)
        return self.transform(feature_df)
    
# Define the ANN model
class EssayScorer(nn.Module):
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.relu(self.layer3(x))
        x = self.output(x)
        return x

def batch_process_essays(essays, func, batch_size=10, desc="Processing"):
    """Process essays in batches to improve performance"""
    results = []
    for i in tqdm(range(0, len(essays), batch_size), desc=desc):
        batch = essays[i:i + batch_size]
        batch_results = [func(essay) for essay in batch]
        results.extend(batch_results)
    return results

def calculate_error_penalties(essay):
    """Optimized error penalty calculation"""
    try:
        cleaned_text = re.sub(r'\s+', ' ', essay).strip()
        words = cleaned_text.split()
        
        if not words:
            return {
                'spelling_penalty': 1.0,
                'grammar_penalty': 1.0,
                'total_penalty': 1.0
            }
        
        # Batch process spelling checks
        misspelled = spell_checker.unknown(words)
        spelling_errors = len(misspelled)
        
        # Limit grammar check to first 1000 characters to improve speed
        truncated_text = cleaned_text[:1000]
        grammar_matches = language_tool.check(truncated_text)
        grammar_errors = len(grammar_matches)
        
        word_count = len(words)
        spelling_penalty = min(1.0, spelling_errors / (word_count * 0.1))
        grammar_penalty = min(1.0, grammar_errors / (word_count * 0.05))
        total_penalty = (spelling_penalty * 0.6) + (grammar_penalty * 0.4)
        
        return {
            'spelling_penalty': spelling_penalty,
            'grammar_penalty': grammar_penalty,
            'total_penalty': total_penalty
        }
    except Exception as e:
        logging.error(f"Error in calculate_error_penalties: {str(e)}")
        return {'spelling_penalty': 0.5, 'grammar_penalty': 0.5, 'total_penalty': 0.5}

def batch_encode_texts(texts, batch_size=32):
    """Batch encode texts using sentence transformer"""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embeddings = sbert_model.encode(batch)
        all_embeddings.extend(embeddings)
    return all_embeddings

def extract_features_with_topic_and_errors(essay, topic):
    """Optimized feature extraction"""
    try:
        # Process with spaCy
        doc = nlp(essay)
        
        # Basic text statistics
        word_count = len([token.text for token in doc if token.is_alpha])
        sentence_count = len(list(doc.sents))
        avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
        unique_words = len(set(token.text.lower() for token in doc if token.is_alpha))
        lexical_density = unique_words / word_count if word_count > 0 else 0
        ttr = unique_words / word_count if word_count > 0 else 0
        
        # POS counting
        pos_count = {pos: len([token for token in doc if token.pos_ == pos]) for pos in ['NOUN', 'VERB']}
        
        # Readability scores (cached internally by textstat)
        flesch_kincaid = textstat.flesch_kincaid_grade(essay)
        dale_chall = textstat.dale_chall_readability_score(essay)
        
        # Sentiment analysis
        sentiment = sentiment_analyzer.polarity_scores(essay)['compound']
        
        # Simplified coherence calculation
        sentences = [sent.text for sent in doc.sents]
        if len(sentences) > 1:
            # Only process first 5 sentences for speed
            sentences = sentences[:5]
            embeddings = sbert_model.encode(sentences)
            coherence_scores = [
                np.dot(embeddings[i], embeddings[i + 1]) /
                (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]))
                for i in range(len(embeddings) - 1)
            ]
            coherence = np.mean(coherence_scores) if coherence_scores else 0
        else:
            coherence = 0
        
        # Calculate topic relevance
        essay_embedding = sbert_model.encode([essay])[0]
        topic_embedding = sbert_model.encode([topic])[0]
        topic_relevance = np.dot(essay_embedding, topic_embedding) / (
            np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
        )
        
        # Get error metrics
        error_metrics = calculate_error_penalties(essay)
        
        return {
            'word_count': word_count,
            'avg_sentence_length': avg_sentence_length,
            'unique_words': unique_words,
            'lexical_density': lexical_density,
            'ttr': ttr,
            'nouns': pos_count['NOUN'],
            'verbs': pos_count['VERB'],
            'flesch_kincaid': flesch_kincaid,
            'dale_chall': dale_chall,
            'sentiment': sentiment,
            'coherence': coherence,
            'topic_relevance': topic_relevance,
            'spelling_penalty': error_metrics['spelling_penalty'],
            'grammar_penalty': error_metrics['grammar_penalty'],
            'total_error_penalty': error_metrics['total_penalty']
        }
    except Exception as e:
        logging.error(f"Error in feature extraction: {str(e)}")
        return {feature: 0 for feature in [
            'word_count', 'avg_sentence_length', 'unique_words', 'lexical_density',
            'ttr', 'nouns', 'verbs', 'flesch_kincaid', 'dale_chall', 'sentiment',
            'coherence', 'topic_relevance', 'spelling_penalty',
            'grammar_penalty', 'total_error_penalty'
        ]}

def process_essay_batch(args):
    """Process a batch of essays"""
    essays, topics = args
    return [extract_features_with_topic_and_errors(essay, topic) 
            for essay, topic in zip(essays, topics)]

def main():
    try:
        # Load data (remove score column if exists)
        print("Loading data...")
        df = pd.read_csv('essays.csv')
        if 'Score' in df.columns:
            df = df.drop(columns=['Score'])
        total_essays = len(df)
        print(f"Loaded {total_essays} essays successfully!")

        # Process essays in parallel batches
        print("\nExtracting features...")
        batch_size = 10
        num_workers = min(os.cpu_count(), 4)  # Limit to 4 workers max
        
        essay_batches = [
            (df['Essay Answer'][i:i + batch_size].tolist(),
             df['Essay Question'][i:i + batch_size].tolist())
            for i in range(0, len(df), batch_size)
        ]
        
        all_features = []
        processed_essays = 0
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(process_essay_batch, batch) 
                      for batch in essay_batches]
            
            progress_bar = tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Processing essay batches"
            )
            
            for future in progress_bar:
                batch_features = future.result()
                all_features.extend(batch_features)
                processed_essays += len(batch_features)
                progress_bar.set_postfix({
                    'essays_processed': f'{processed_essays}/{total_essays}',
                    'completion': f'{(processed_essays/total_essays)*100:.1f}%'
                })
        
        print(f"\nProcessed {processed_essays} essays in {len(essay_batches)} batches")
        
        X = pd.DataFrame(all_features)
        # Generate scores using heuristic formula
        print("\nGenerating heuristic scores...")

        """ 
        Creates a heuristic scoring system using:
        Topic relevance (40% weight)
        Text coherence (30% weight)
        Error penalties (30% weight)
        Readability scores (additional 20% weight)
        """
        
        # Calculate components with error handling
        def safe_get(col, default=0):
            return X[col].clip(0, 1).fillna(default)
        
        # Calculate components
        topic_component = safe_get('topic_relevance') * 4
        coherence_component = safe_get('coherence') * 3
        error_component = (1 - safe_get('total_error_penalty')) * 3
        readability_component = ((safe_get('flesch_kincaid', 0)/12.0) + 
                                (1 - safe_get('dale_chall', 0)/10.0)) / 2 * 2
        
        # Combine components with weights
        raw_scores = (
            topic_component + 
            coherence_component + 
            error_component + 
            readability_component
        )
        
        # Scale to 1-10 range with floor/ceiling
        y = raw_scores.clip(0, 10)
        y = ((y - y.min()) / (y.max() - y.min())) * 9 + 1
        y = round(y * 2) / 2
        
        print("Score distribution summary:")
        print(pd.DataFrame(y).describe())
        print("\nValue count distribution")
        print(pd.DataFrame(y).value_counts().sort_index())

        # Initialize feature normalizer
        print("\nNormalizing features...")
        normalizer = FeatureNormalizer()

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Normalize features
        X_train_normalized = normalizer.fit_transform(X_train)
        X_test_normalized = normalizer.transform(X_test)

        # Print feature ranges
        print("\nFeature ranges after normalization (training set):")
        for column in X_train_normalized.columns:
            min_val = X_train_normalized[column].min()
            max_val = X_train_normalized[column].max()
            mean_val = X_train_normalized[column].mean()
            print(f"{column:20} - Min: {min_val:.2f}, Max: {max_val:.2f}, Mean: {mean_val:.2f}")

        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train_normalized.values)
        X_test_tensor = torch.FloatTensor(X_test_normalized.values)
        y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
        y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

        # Initialize model
        model = EssayScorer(input_size=X_train.shape[1])
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Training loop
        print("\nStarting training...")
        num_epochs = 500
        batch_size = 32
        
        progress_bar = tqdm(range(num_epochs), desc="Training")
        for epoch in progress_bar:
            model.train()
            total_loss = 0
            num_batches = 0
            
            for i in range(0, len(X_train_tensor), batch_size):
                batch_X = X_train_tensor[i:i + batch_size]
                batch_y = y_train_tensor[i:i + batch_size]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                num_batches += 1
            
            avg_loss = total_loss / num_batches
            progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})

        # Evaluate model
        print("\nEvaluating model...")
        model.eval()
        with torch.no_grad():
            y_pred = model(X_test_tensor)
            
            mse = mean_squared_error(y_test_tensor.numpy(), y_pred.numpy())
            rmse = math.sqrt(mse)
            mae = mean_absolute_error(y_test_tensor.numpy(), y_pred.numpy())
            
            print("\nTest Metrics:")
            print(f"MSE: {mse:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f"MAE: {mae:.4f}")

        # Save model
        print("\nSaving model...")
        torch.save({
            'model_state_dict': model.state_dict(),
            'normalizer': normalizer,
            'feature_columns': list(X_train.columns)
        }, 'normalized_essay_scorer.pth')
        print("Model and normalizer saved successfully!")

    except Exception as e:
        logging.error(f"Error in main function: {str(e)}")
        raise

if __name__ == "__main__":
    main()

Loading NLP models...
Models loaded successfully!
Loading data...
Loaded 2979 essays successfully!

Extracting features...


Processing essay batches: 100%|██████████| 298/298 [03:31<00:00,  1.41it/s, essays_processed=2979/2979, completion=100.0%]



Processed 2979 essays in 298 batches

Generating heuristic scores...
Score distribution summary:
                 0
count  2979.000000
mean      6.298899
std       1.277636
min       1.000000
25%       5.548284
50%       6.323151
75%       7.144133
max      10.000000
0        
1.000000     1
1.062684     1
1.577422     1
1.673869     1
1.782996     1
            ..
9.713293     1
9.800418     1
9.850353     1
9.964078     1
10.000000    1
Name: count, Length: 2979, dtype: int64

Normalizing features...

Feature ranges after normalization (training set):
word_count           - Min: 1.00, Max: 10.00, Mean: 4.24
avg_sentence_length  - Min: 1.00, Max: 10.00, Mean: 1.86
unique_words         - Min: 1.00, Max: 10.00, Mean: 4.90
lexical_density      - Min: 1.00, Max: 10.00, Mean: 4.22
ttr                  - Min: 1.00, Max: 10.00, Mean: 4.22
nouns                - Min: 1.00, Max: 10.00, Mean: 4.16
verbs                - Min: 1.00, Max: 10.00, Mean: 3.52
flesch_kincaid       - Min: 1.00, Max: 1

Training: 100%|██████████| 500/500 [03:22<00:00,  2.46it/s, loss=0.0082]


Evaluating model...

Test Metrics:
MSE: 0.3524
RMSE: 0.5937
MAE: 0.5397

Saving model...
Model and normalizer saved successfully!


### ANN with Topic and Essay Embeddings with SpellCheck, Grammer and newly Graded Scores (rounded to 0.5 or nearest whole number)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"  # Changed to true for batch processing

import spacy
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import math
from spellchecker import SpellChecker
import re
from tqdm import tqdm
import warnings
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
import language_tool_python

# Configure warnings and logging
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
logging.basicConfig(level=logging.ERROR)

# Initialize shared resources
print("Loading NLP models...")
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
spell_checker = SpellChecker()
language_tool = language_tool_python.LanguageTool('en-US')
print("Models loaded successfully!")

class FeatureNormalizer:
    def __init__(self):
        self.scalers = {}
        
    def fit(self, feature_df):
        """Fit scalers for each feature"""
        for column in feature_df.columns:
            self.scalers[column] = MinMaxScaler(feature_range=(1, 10))
            self.scalers[column].fit(feature_df[column].values.reshape(-1, 1))
    
    def transform(self, feature_df):
        """Transform features to range 1-10"""
        normalized_df = pd.DataFrame()
        for column in feature_df.columns:
            normalized_values = self.scalers[column].transform(
                feature_df[column].values.reshape(-1, 1)
            )
            normalized_df[column] = normalized_values.flatten()
        return normalized_df
    
    def fit_transform(self, feature_df):
        """Fit and transform features"""
        self.fit(feature_df)
        return self.transform(feature_df)
    
# Define the ANN model
class EssayScorer(nn.Module):
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.relu(self.layer3(x))
        x = self.output(x)
        return x

def batch_process_essays(essays, func, batch_size=10, desc="Processing"):
    """Process essays in batches to improve performance"""
    results = []
    for i in tqdm(range(0, len(essays), batch_size), desc=desc):
        batch = essays[i:i + batch_size]
        batch_results = [func(essay) for essay in batch]
        results.extend(batch_results)
    return results

def calculate_error_penalties(essay):
    """Optimized error penalty calculation"""
    try:
        cleaned_text = re.sub(r'\s+', ' ', essay).strip()
        words = cleaned_text.split()
        
        if not words:
            return {
                'spelling_penalty': 1.0,
                'grammar_penalty': 1.0,
                'total_penalty': 1.0
            }
        
        # Batch process spelling checks
        misspelled = spell_checker.unknown(words)
        spelling_errors = len(misspelled)
        
        # Limit grammar check to first 1000 characters to improve speed
        truncated_text = cleaned_text[:1000]
        grammar_matches = language_tool.check(truncated_text)
        grammar_errors = len(grammar_matches)
        
        word_count = len(words)
        spelling_penalty = min(1.0, spelling_errors / (word_count * 0.1))
        grammar_penalty = min(1.0, grammar_errors / (word_count * 0.05))
        total_penalty = (spelling_penalty * 0.6) + (grammar_penalty * 0.4)
        
        return {
            'spelling_penalty': spelling_penalty,
            'grammar_penalty': grammar_penalty,
            'total_penalty': total_penalty
        }
    except Exception as e:
        logging.error(f"Error in calculate_error_penalties: {str(e)}")
        return {'spelling_penalty': 0.5, 'grammar_penalty': 0.5, 'total_penalty': 0.5}

def batch_encode_texts(texts, batch_size=32):
    """Batch encode texts using sentence transformer"""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        embeddings = sbert_model.encode(batch)
        all_embeddings.extend(embeddings)
    return all_embeddings

def extract_features_with_topic_and_errors(essay, topic):
    """Optimized feature extraction"""
    try:
        # Process with spaCy
        doc = nlp(essay)
        
        # Basic text statistics
        word_count = len([token.text for token in doc if token.is_alpha])
        sentence_count = len(list(doc.sents))
        avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
        unique_words = len(set(token.text.lower() for token in doc if token.is_alpha))
        lexical_density = unique_words / word_count if word_count > 0 else 0
        ttr = unique_words / word_count if word_count > 0 else 0
        
        # POS counting
        pos_count = {pos: len([token for token in doc if token.pos_ == pos]) for pos in ['NOUN', 'VERB']}
        
        # Readability scores (cached internally by textstat)
        flesch_kincaid = textstat.flesch_kincaid_grade(essay)
        dale_chall = textstat.dale_chall_readability_score(essay)
        
        # Sentiment analysis
        sentiment = sentiment_analyzer.polarity_scores(essay)['compound']
        
        # Simplified coherence calculation
        sentences = [sent.text for sent in doc.sents]
        if len(sentences) > 1:
            # Only process first 5 sentences for speed
            sentences = sentences[:5]
            embeddings = sbert_model.encode(sentences)
            coherence_scores = [
                np.dot(embeddings[i], embeddings[i + 1]) /
                (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]))
                for i in range(len(embeddings) - 1)
            ]
            coherence = np.mean(coherence_scores) if coherence_scores else 0
        else:
            coherence = 0
        
        # Calculate topic relevance
        essay_embedding = sbert_model.encode([essay])[0]
        topic_embedding = sbert_model.encode([topic])[0]
        topic_relevance = np.dot(essay_embedding, topic_embedding) / (
            np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
        )
        
        # Get error metrics
        error_metrics = calculate_error_penalties(essay)
        
        return {
            'word_count': word_count,
            'avg_sentence_length': avg_sentence_length,
            'unique_words': unique_words,
            'lexical_density': lexical_density,
            'ttr': ttr,
            'nouns': pos_count['NOUN'],
            'verbs': pos_count['VERB'],
            'flesch_kincaid': flesch_kincaid,
            'dale_chall': dale_chall,
            'sentiment': sentiment,
            'coherence': coherence,
            'topic_relevance': topic_relevance,
            'spelling_penalty': error_metrics['spelling_penalty'],
            'grammar_penalty': error_metrics['grammar_penalty'],
            'total_error_penalty': error_metrics['total_penalty']
        }
    except Exception as e:
        logging.error(f"Error in feature extraction: {str(e)}")
        return {feature: 0 for feature in [
            'word_count', 'avg_sentence_length', 'unique_words', 'lexical_density',
            'ttr', 'nouns', 'verbs', 'flesch_kincaid', 'dale_chall', 'sentiment',
            'coherence', 'topic_relevance', 'spelling_penalty',
            'grammar_penalty', 'total_error_penalty'
        ]}

def process_essay_batch(args):
    """Process a batch of essays"""
    essays, topics = args
    return [extract_features_with_topic_and_errors(essay, topic) 
            for essay, topic in zip(essays, topics)]

def main():
    try:
        # Load data (remove score column if exists)
        print("Loading data...")
        df = pd.read_csv('essays.csv')
        if 'Score' in df.columns:
            df = df.drop(columns=['Score'])
        total_essays = len(df)
        print(f"Loaded {total_essays} essays successfully!")

        # Process essays in parallel batches
        print("\nExtracting features...")
        batch_size = 10
        num_workers = min(os.cpu_count(), 4)  # Limit to 4 workers max
        
        essay_batches = [
            (df['Essay Answer'][i:i + batch_size].tolist(),
             df['Essay Question'][i:i + batch_size].tolist())
            for i in range(0, len(df), batch_size)
        ]
        
        all_features = []
        processed_essays = 0
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(process_essay_batch, batch) 
                      for batch in essay_batches]
            
            progress_bar = tqdm(
                as_completed(futures),
                total=len(futures),
                desc="Processing essay batches"
            )
            
            for future in progress_bar:
                batch_features = future.result()
                all_features.extend(batch_features)
                processed_essays += len(batch_features)
                progress_bar.set_postfix({
                    'essays_processed': f'{processed_essays}/{total_essays}',
                    'completion': f'{(processed_essays/total_essays)*100:.1f}%'
                })
        
        print(f"\nProcessed {processed_essays} essays in {len(essay_batches)} batches")
        
        X = pd.DataFrame(all_features)
        # Generate scores using heuristic formula
        print("\nGenerating heuristic scores...")

        """ 
        Creates a heuristic scoring system using:
        Topic relevance (40% weight)
        Text coherence (30% weight)
        Error penalties (30% weight)
        Readability scores (additional 20% weight)
        """
        
        # Calculate components with error handling
        def safe_get(col, default=0):
            return X[col].clip(0, 1).fillna(default)
        
        # Calculate components
        topic_component = safe_get('topic_relevance') * 4
        coherence_component = safe_get('coherence') * 3
        error_component = (1 - safe_get('total_error_penalty')) * 3
        readability_component = ((safe_get('flesch_kincaid', 0)/12.0) + 
                                (1 - safe_get('dale_chall', 0)/10.0)) / 2 * 2
        
        # Combine components with weights
        raw_scores = (
            topic_component + 
            coherence_component + 
            error_component + 
            readability_component
        )
        
        # Scale to 1-10 range with floor/ceiling
        y = raw_scores.clip(0, 10)
        y = ((y - y.min()) / (y.max() - y.min())) * 9 + 1
        y = round(y * 2) / 2
        
        print("Score distribution summary:")
        print(pd.DataFrame(y).describe())
        print("\nValue count distribution")
        print(pd.DataFrame(y).value_counts().sort_index())

        # Initialize feature normalizer
        print("\nNormalizing features...")
        normalizer = FeatureNormalizer()

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Normalize features
        X_train_normalized = normalizer.fit_transform(X_train)
        X_test_normalized = normalizer.transform(X_test)

        # Print feature ranges
        print("\nFeature ranges after normalization (training set):")
        for column in X_train_normalized.columns:
            min_val = X_train_normalized[column].min()
            max_val = X_train_normalized[column].max()
            mean_val = X_train_normalized[column].mean()
            print(f"{column:20} - Min: {min_val:.2f}, Max: {max_val:.2f}, Mean: {mean_val:.2f}")

        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train_normalized.values)
        X_test_tensor = torch.FloatTensor(X_test_normalized.values)
        y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
        y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

        # Initialize model
        model = EssayScorer(input_size=X_train.shape[1])
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Training loop
        print("\nStarting training...")
        num_epochs = 500
        batch_size = 32
        
        progress_bar = tqdm(range(num_epochs), desc="Training")
        for epoch in progress_bar:
            model.train()
            total_loss = 0
            num_batches = 0
            
            for i in range(0, len(X_train_tensor), batch_size):
                batch_X = X_train_tensor[i:i + batch_size]
                batch_y = y_train_tensor[i:i + batch_size]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                num_batches += 1
            
            avg_loss = total_loss / num_batches
            progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})

        # Evaluate model
        print("\nEvaluating model...")
        model.eval()
        with torch.no_grad():
            y_pred = model(X_test_tensor)
            
            mse = mean_squared_error(y_test_tensor.numpy(), y_pred.numpy())
            rmse = math.sqrt(mse)
            mae = mean_absolute_error(y_test_tensor.numpy(), y_pred.numpy())
            
            print("\nTest Metrics:")
            print(f"MSE: {mse:.4f}")
            print(f"RMSE: {rmse:.4f}")
            print(f"MAE: {mae:.4f}")

        # Save model
        print("\nSaving model...")
        torch.save({
            'model_state_dict': model.state_dict(),
            'normalizer': normalizer,
            'feature_columns': list(X_train.columns)
        }, 'normalized_essay_scorer.pth')
        print("Model and normalizer saved successfully!")

    except Exception as e:
        logging.error(f"Error in main function: {str(e)}")
        raise

if __name__ == "__main__":
    main()

Loading NLP models...
Models loaded successfully!
Loading data...
Loaded 2979 essays successfully!

Extracting features...


Processing essay batches: 100%|██████████| 298/298 [02:29<00:00,  1.99it/s, essays_processed=2979/2979, completion=100.0%]



Processed 2979 essays in 298 batches

Generating heuristic scores...
Score distribution summary:
                 0
count  2979.000000
mean      6.304129
std       1.285537
min       1.000000
25%       5.500000
50%       6.500000
75%       7.000000
max      10.000000

Value count distribution
0   
1.0       2
1.5       2
2.0      10
2.5      11
3.0      23
3.5      49
4.0      90
4.5     138
5.0     210
5.5     364
6.0     516
6.5     466
7.0     428
7.5     320
8.0     181
8.5      99
9.0      47
9.5      19
10.0      4
Name: count, dtype: int64

Normalizing features...

Feature ranges after normalization (training set):
word_count           - Min: 1.00, Max: 10.00, Mean: 4.24
avg_sentence_length  - Min: 1.00, Max: 10.00, Mean: 1.85
unique_words         - Min: 1.00, Max: 10.00, Mean: 4.90
lexical_density      - Min: 1.00, Max: 10.00, Mean: 4.23
ttr                  - Min: 1.00, Max: 10.00, Mean: 4.23
nouns                - Min: 1.00, Max: 10.00, Mean: 4.16
verbs                - Min:

Training: 100%|██████████| 500/500 [02:57<00:00,  2.82it/s, loss=0.0312]


Evaluating model...

Test Metrics:
MSE: 0.6511
RMSE: 0.8069
MAE: 0.7172

Saving model...
Model and normalizer saved successfully!


### ANN with Topic and Essay Embeddings with SpellCheck, Grammer and newly Graded Scores using BERT

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
from sklearn.preprocessing import MinMaxScaler
import spacy
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import logging
import re
from concurrent.futures import ThreadPoolExecutor

# Configure logging
logging.basicConfig(level=logging.ERROR)

# Initialize components
print("Loading models...")
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Load BERT essay scoring model
BERT_MODEL_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME)
bert_scorer = pipeline(
    "text-classification",
    model=bert_model,
    tokenizer=bert_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)
print("All models loaded successfully!")

# Helper functions
def get_bert_score(essay):
    """Get normalized score from BERT model (0-1 to 1-10)"""
    try:
        result = bert_scorer(essay[:512], truncation=True)[0]
        return round(result['score'] * 9 + 1, 2)  # Scale to 1-10
    except Exception as e:
        logging.error(f"BERT scoring failed: {str(e)}")
        return 5.0  # Fallback score

def batch_bert_scoring(essays, batch_size=8):
    """Process essays in batches for BERT scoring"""
    scores = []
    for i in tqdm(range(0, len(essays), batch_size), desc="BERT Scoring"):
        batch = essays[i:i+batch_size]
        batch = [e[:512] for e in batch]  # Truncate to 512 tokens
        scores.extend([round(r['score'] * 9 + 1, 2) for r in bert_scorer(batch)])
    return scores

def extract_linguistic_features(essay):
    """Enhanced linguistic feature extraction"""
    doc = nlp(essay)
    
    # Readability metrics
    readability = {
        'flesch_kincaid': textstat.flesch_kincaid_grade(essay),
        'dale_chall': textstat.dale_chall_readability_score(essay),
        'coleman_liau': textstat.coleman_liau_index(essay)
    }
    
    # Coherence calculation
    sentences = [sent.text for sent in doc.sents][:5]
    if len(sentences) > 1:
        embeddings = sbert_model.encode(sentences)
        coherence = np.mean([
            np.dot(embeddings[i], embeddings[i+1]) / 
            (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1]))
            for i in range(len(embeddings)-1)
        ])
    else:
        coherence = 0
    
    return {
        'word_count': len([t for t in doc if t.is_alpha]),
        'avg_sentence_length': readability['flesch_kincaid'],
        'unique_words': len(set(t.text.lower() for t in doc if t.is_alpha)),
        'ttr': len(set(t.text.lower() for t in doc)) / len(doc) if doc else 0,
        'coherence': coherence,
        **readability,
        'sentiment': SentimentIntensityAnalyzer().polarity_scores(essay)['compound']
    }

class EssayDataset(torch.utils.data.Dataset):
    """Custom dataset for essay scoring"""
    def __init__(self, features, scores):
        self.features = torch.FloatTensor(features.values)
        self.scores = torch.FloatTensor(scores.values.reshape(-1, 1))
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.scores[idx]

class EssayScorer(nn.Module):
    """Neural network with BERT + linguistic features"""
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.fc(x)

class FeatureNormalizer:
    """Enhanced feature normalization"""
    def __init__(self):
        self.scaler = MinMaxScaler()
    
    def fit_transform(self, X):
        return pd.DataFrame(self.scaler.fit_transform(X), columns=X.columns)
    
    def transform(self, X):
        return pd.DataFrame(self.scaler.transform(X), columns=X.columns)

def main():
    # Load data
    df = pd.read_csv('essays.csv')
    
    # Generate BERT scores
    print("\nGenerating BERT scores...")
    df['bert_score'] = batch_bert_scoring(df['Essay Answer'].tolist())
    print("\nScore values distribution:")
    print(df['bert_score'].value_counts().sort_index())
    
    
    # Extract linguistic features
    print("\nExtracting linguistic features...")
    with ThreadPoolExecutor(max_workers=4) as executor:
        features = list(tqdm(
            executor.map(extract_linguistic_features, df['Essay Answer']),
            total=len(df),
            desc="Feature Extraction"
        ))
    
    # Combine features
    X = pd.DataFrame(features)
    X['bert_score'] = df['bert_score']
    y = df['bert_score']  # Using BERT scores as targets
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Normalize features
    normalizer = FeatureNormalizer()
    X_train = normalizer.fit_transform(X_train)
    X_test = normalizer.transform(X_test)
    
    # Create datasets
    train_dataset = EssayDataset(X_train, y_train)
    test_dataset = EssayDataset(X_test, y_test)
    
    # Initialize model
    model = EssayScorer(input_size=X_train.shape[1])
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.HuberLoss()
    
    # Training loop
    print("\nTraining model...")
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=32, shuffle=True
    )
    
    for epoch in range(50):
        model.train()
        total_loss = 0
        for batch_features, batch_scores in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_scores)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)
        predictions = []
        for batch_features, _ in test_loader:
            preds = model(batch_features).numpy().flatten()
            predictions.extend(preds)
        
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test,predictions)
        
        print(f"\nFinal Metrics:")
        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"R2 Score: {r2:.2f}")
    
    # Save model
    torch.save({
        'model_state': model.state_dict(),
        'normalizer': normalizer,
        'feature_columns': X.columns.tolist()
    }, "bert_enhanced_scorer.pth")

if __name__ == "__main__":
    main()

Loading models...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


All models loaded successfully!

Generating BERT scores...


BERT Scoring: 100%|██████████| 373/373 [00:41<00:00,  8.98it/s]



Score values distribution:
bert_score
5.51    3
5.52    3
5.53    2
5.54    2
5.55    3
       ..
6.91    1
6.92    1
6.95    2
7.00    1
7.21    1
Name: count, Length: 138, dtype: int64

Extracting linguistic features...


...

Epoch 48 | Avg Loss: 0.0949


Epoch 49: 100%|██████████| 75/75 [00:00<00:00, 982.96it/s]


Epoch 49 | Avg Loss: 0.0922


Epoch 50: 100%|██████████| 75/75 [00:00<00:00, 971.25it/s]

Epoch 50 | Avg Loss: 0.0909

Final Metrics:
MAE: 0.08
RMSE: 0.11
R2 Score: 0.78


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
from sklearn.preprocessing import MinMaxScaler
import spacy
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import logging
import re
from concurrent.futures import ThreadPoolExecutor

# Configure logging
logging.basicConfig(level=logging.ERROR)

# Initialize components
print("Loading models...")
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Load BERT essay scoring model
BERT_MODEL_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME)
bert_scorer = pipeline(
    "text-classification",
    model=bert_model,
    tokenizer=bert_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)
print("All models loaded successfully!")

# Helper functions
def get_bert_score(essay):
    """Get normalized score from BERT model (0-1 to 1-10)"""
    try:
        result = bert_scorer(essay[:512], truncation=True)[0]
        return round(result['score'] * 9 + 1, 2)  # Scale to 1-10
    except Exception as e:
        logging.error(f"BERT scoring failed: {str(e)}")
        return 5.0  # Fallback score

def batch_bert_scoring(essays, batch_size=8):
    """Process essays in batches for BERT scoring"""
    scores = []
    for i in tqdm(range(0, len(essays), batch_size), desc="BERT Scoring"):
        batch = essays[i:i+batch_size]
        batch = [e[:512] for e in batch]  # Truncate to 512 tokens
        scores.extend([round(r['score'] * 9 + 1, 2) for r in bert_scorer(batch)])
    return scores

def extract_linguistic_features(essay):
    """Enhanced linguistic feature extraction"""
    doc = nlp(essay)
    
    # Readability metrics
    readability = {
        'flesch_kincaid': textstat.flesch_kincaid_grade(essay),
        'dale_chall': textstat.dale_chall_readability_score(essay),
        'coleman_liau': textstat.coleman_liau_index(essay)
    }
    
    # Coherence calculation
    sentences = [sent.text for sent in doc.sents][:5]
    if len(sentences) > 1:
        embeddings = sbert_model.encode(sentences)
        coherence = np.mean([
            np.dot(embeddings[i], embeddings[i+1]) / 
            (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1]))
            for i in range(len(embeddings)-1)
        ])
    else:
        coherence = 0
    
    return {
        'word_count': len([t for t in doc if t.is_alpha]),
        'avg_sentence_length': readability['flesch_kincaid'],
        'unique_words': len(set(t.text.lower() for t in doc if t.is_alpha)),
        'ttr': len(set(t.text.lower() for t in doc)) / len(doc) if doc else 0,
        'coherence': coherence,
        **readability,
        'sentiment': SentimentIntensityAnalyzer().polarity_scores(essay)['compound']
    }

class EssayDataset(torch.utils.data.Dataset):
    """Custom dataset for essay scoring"""
    def __init__(self, features, scores):
        self.features = torch.FloatTensor(features.values)
        self.scores = torch.FloatTensor(scores.values.reshape(-1, 1))
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.scores[idx]

class EssayScorer(nn.Module):
    """Neural network with BERT + linguistic features"""
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.fc(x)

class FeatureNormalizer:
    """Enhanced feature normalization"""
    def __init__(self):
        self.scaler = MinMaxScaler()
    
    def fit_transform(self, X):
        return pd.DataFrame(self.scaler.fit_transform(X), columns=X.columns)
    
    def transform(self, X):
        return pd.DataFrame(self.scaler.transform(X), columns=X.columns)

def main():
    # Load data
    df = pd.read_csv('essays.csv')
    
    # Generate BERT scores
    print("\nGenerating BERT scores...")
    df['bert_score'] = batch_bert_scoring(df['Essay Answer'].tolist())
    
    
    # Extract linguistic features
    print("\nExtracting linguistic features...")
    with ThreadPoolExecutor(max_workers=4) as executor:
        features = list(tqdm(
            executor.map(extract_linguistic_features, df['Essay Answer']),
            total=len(df),
            desc="Feature Extraction"
        ))
    
    # Combine features
    X = pd.DataFrame(features)
    X['bert_score'] = df['bert_score']
    Y = df['bert_score']  # Using BERT scores as targets
    y = round(Y * 2) / 2

    print("\nScore values distribution:")
    print(df['bert_score'].value_counts().sort_index())
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Normalize features
    normalizer = FeatureNormalizer()
    X_train = normalizer.fit_transform(X_train)
    X_test = normalizer.transform(X_test)
    
    # Create datasets
    train_dataset = EssayDataset(X_train, y_train)
    test_dataset = EssayDataset(X_test, y_test)
    
    # Initialize model
    model = EssayScorer(input_size=X_train.shape[1])
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.HuberLoss()
    
    # Training loop
    print("\nTraining model...")
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=32, shuffle=True
    )
    
    for epoch in range(50):
        model.train()
        total_loss = 0
        for batch_features, batch_scores in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_scores)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)
        predictions = []
        for batch_features, _ in test_loader:
            preds = model(batch_features).numpy().flatten()
            predictions.extend(preds)
        
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test,predictions)
        
        print(f"\nFinal Metrics:")
        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"R2 Score: {r2:.2f}")
    
    # Save model
    torch.save({
        'model_state': model.state_dict(),
        'normalizer': normalizer,
        'feature_columns': X.columns.tolist()
    }, "bert_enhanced_scorer.pth")

if __name__ == "__main__":
    main()

Loading models...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


All models loaded successfully!

Generating BERT scores...


BERT Scoring: 100%|██████████| 373/373 [00:41<00:00,  9.01it/s]



Extracting linguistic features...


...

Epoch 48 | Avg Loss: 0.1268


Epoch 49: 100%|██████████| 75/75 [00:00<00:00, 1499.47it/s]


Epoch 49 | Avg Loss: 0.1337


Epoch 50: 100%|██████████| 75/75 [00:00<00:00, 1501.58it/s]

Epoch 50 | Avg Loss: 0.1281

Final Metrics:
MAE: 0.16
RMSE: 0.20
R2 Score: 0.81


In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
from sklearn.preprocessing import MinMaxScaler
import spacy
from textstat import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import logging
import re
from concurrent.futures import ThreadPoolExecutor

# Configure logging
logging.basicConfig(level=logging.ERROR)

# Initialize components
print("Loading models...")
nlp = spacy.load('en_core_web_sm')
sentiment_analyzer = SentimentIntensityAnalyzer()
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Load BERT essay scoring model
BERT_MODEL_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_NAME)
bert_scorer = pipeline(
    "text-classification",
    model=bert_model,
    tokenizer=bert_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)
print("All models loaded successfully!")

# Helper functions
def get_bert_score(essay):
    """Get normalized score from BERT model (0-1 to 1-10)"""
    try:
        result = bert_scorer(essay[:512], truncation=True)[0]
        return round(result['score'] * 9 + 1, 2)  # Scale to 1-10
    except Exception as e:
        logging.error(f"BERT scoring failed: {str(e)}")
        return 5.0  # Fallback score

def batch_bert_scoring(essays, batch_size=8):
    """Process essays in batches for BERT scoring"""
    scores = []
    for i in tqdm(range(0, len(essays), batch_size), desc="BERT Scoring"):
        batch = essays[i:i+batch_size]
        batch = [e[:512] for e in batch]  # Truncate to 512 tokens
        scores.extend([round(r['score'] * 9 + 1, 2) for r in bert_scorer(batch)])
    return scores

def extract_features_with_topic_and_errors(essay, topic):
    """Optimized feature extraction"""
    try:
        # Process with spaCy
        doc = nlp(essay)
        
        # Basic text statistics
        word_count = len([token.text for token in doc if token.is_alpha])
        sentence_count = len(list(doc.sents))
        avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
        unique_words = len(set(token.text.lower() for token in doc if token.is_alpha))
        lexical_density = unique_words / word_count if word_count > 0 else 0
        ttr = unique_words / word_count if word_count > 0 else 0
        
        # POS counting
        pos_count = {pos: len([token for token in doc if token.pos_ == pos]) for pos in ['NOUN', 'VERB']}
        
        # Readability scores (cached internally by textstat)
        flesch_kincaid = textstat.flesch_kincaid_grade(essay)
        dale_chall = textstat.dale_chall_readability_score(essay)
        
        # Sentiment analysis
        sentiment = sentiment_analyzer.polarity_scores(essay)['compound']
        
        # Simplified coherence calculation
        sentences = [sent.text for sent in doc.sents]
        if len(sentences) > 1:
            # Only process first 5 sentences for speed
            sentences = sentences[:5]
            embeddings = sbert_model.encode(sentences)
            coherence_scores = [
                np.dot(embeddings[i], embeddings[i + 1]) /
                (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]))
                for i in range(len(embeddings) - 1)
            ]
            coherence = np.mean(coherence_scores) if coherence_scores else 0
        else:
            coherence = 0
        
        # Calculate topic relevance
        essay_embedding = sbert_model.encode([essay])[0]
        topic_embedding = sbert_model.encode([topic])[0]
        topic_relevance = np.dot(essay_embedding, topic_embedding) / (
            np.linalg.norm(essay_embedding) * np.linalg.norm(topic_embedding)
        )
        
        # Get error metrics
        error_metrics = calculate_error_penalties(essay)
        
        return {
            'word_count': word_count,
            'avg_sentence_length': avg_sentence_length,
            'unique_words': unique_words,
            'lexical_density': lexical_density,
            'ttr': ttr,
            'nouns': pos_count['NOUN'],
            'verbs': pos_count['VERB'],
            'flesch_kincaid': flesch_kincaid,
            'dale_chall': dale_chall,
            'sentiment': sentiment,
            'coherence': coherence,
            'topic_relevance': topic_relevance,
            'spelling_penalty': error_metrics['spelling_penalty'],
            'grammar_penalty': error_metrics['grammar_penalty'],
            'total_error_penalty': error_metrics['total_penalty']
        }
    except Exception as e:
        logging.error(f"Error in feature extraction: {str(e)}")
        return {feature: 0 for feature in [
            'word_count', 'avg_sentence_length', 'unique_words', 'lexical_density',
            'ttr', 'nouns', 'verbs', 'flesch_kincaid', 'dale_chall', 'sentiment',
            'coherence', 'topic_relevance', 'spelling_penalty',
            'grammar_penalty', 'total_error_penalty'
        ]}

class EssayDataset(torch.utils.data.Dataset):
    """Custom dataset for essay scoring"""
    def __init__(self, features, scores):
        self.features = torch.FloatTensor(features.values)
        self.scores = torch.FloatTensor(scores.values.reshape(-1, 1))
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.scores[idx]

class EssayScorer(nn.Module):
    """Neural network with BERT + linguistic features"""
    def __init__(self, input_size):
        super(EssayScorer, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.fc(x)

class FeatureNormalizer:
    """Enhanced feature normalization"""
    def __init__(self):
        self.scaler = MinMaxScaler()
    
    def fit_transform(self, X):
        return pd.DataFrame(self.scaler.fit_transform(X), columns=X.columns)
    
    def transform(self, X):
        return pd.DataFrame(self.scaler.transform(X), columns=X.columns)

def main():
    # Load data
    df = pd.read_csv('essays.csv')
    
    # Generate BERT scores
    print("\nGenerating BERT scores...")
    df['bert_score'] = batch_bert_scoring(df['Essay Answer'].tolist())
    
    # Extract linguistic features
    print("\nExtracting linguistic features...")
    with ThreadPoolExecutor(max_workers=4) as executor:
        features = list(tqdm(
            executor.map(extract_linguistic_features, df['Essay Answer']),
            total=len(df),
            desc="Feature Extraction"
        ))
    
    # Combine features
    X = pd.DataFrame(features)
    X['bert_score'] = df['bert_score']
    y = df['bert_score']  # Using BERT scores as targets
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Normalize features
    normalizer = FeatureNormalizer()
    X_train = normalizer.fit_transform(X_train)
    X_test = normalizer.transform(X_test)
    
    # Create datasets
    train_dataset = EssayDataset(X_train, y_train)
    test_dataset = EssayDataset(X_test, y_test)
    
    # Initialize model
    model = EssayScorer(input_size=X_train.shape[1])
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.HuberLoss()
    
    # Training loop
    print("\nTraining model...")
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=32, shuffle=True
    )
    
    for epoch in range(50):
        model.train()
        total_loss = 0
        for batch_features, batch_scores in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_scores)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)
        predictions = []
        for batch_features, _ in test_loader:
            preds = model(batch_features).numpy().flatten()
            predictions.extend(preds)
        
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test,predictions)
        
        print(f"\nFinal Metrics:")
        print(f"MAE: {mae:.2f}")
        print(f"RMSE: {rmse:.2f}")
        print(f"R2 Score: {r2:.2f}")
    
    # Save model
    torch.save({
        'model_state': model.state_dict(),
        'normalizer': normalizer,
        'feature_columns': X.columns.tolist()
    }, "bert_enhanced_scorer.pth")

if __name__ == "__main__":
    main()

Loading models...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


All models loaded successfully!

Generating BERT scores...


BERT Scoring: 100%|██████████| 373/373 [00:41<00:00,  8.96it/s]



Extracting linguistic features...


...

Epoch 48 | Avg Loss: 0.0918


Epoch 49: 100%|██████████| 75/75 [00:00<00:00, 1254.83it/s]


Epoch 49 | Avg Loss: 0.0845


Epoch 50: 100%|██████████| 75/75 [00:00<00:00, 1257.78it/s]

Epoch 50 | Avg Loss: 0.0887

Final Metrics:
MAE: 0.09
RMSE: 0.11
R2 Score: 0.82
